# 04 — Earth Pulse: AI scenario cards (one click, GPU)

Generates the **three scenario images** (lower / median / upper) of a place, conditioned on
the latest real NASA GIBS tile + the model's predicted NDVI delta.

**Honesty rules (enforced in code):** label is always `AI-generated scenario — not an observation`.
Geography is preserved (img2img conditioning); only land-cover balance shifts. Never a forecast photo.

## What you need
1. **Runtime → Change runtime type → GPU** (T4 free tier is fine)
2. Just press **Run all** — no API keys, no logins anywhere.

Total time ≈ 10–15 min (most of it is the ~4 GB model download).

In [ ]:
import subprocess, sys, os, json, glob, zipfile
subprocess.run(['nvidia-smi'], check=False)

## 1. Clone Earth Pulse

In [ ]:
%cd /content
if not os.path.exists('/content/earth-pulse'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/cyberhatc/earth-pulse.git'], check=True)
%cd /content/earth-pulse

## 2. Install dependencies
(CPU wheels first for diffusers bits; torch is already present on Colab GPU images)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.30', 'transformers>=4.44', 'accelerate>=0.33',
    'pillow', 'requests', 'numpy', 'scipy', 'fastapi', 'uvicorn',
    'matplotlib', 'tqdm'], check=True)
print('deps installed')

## 3. Sanity check (build demo store + forecaster + verify)

In [ ]:
subprocess.run([sys.executable, 'pipeline/run_pipeline.py', '--sample'], check=True)
subprocess.run([sys.executable, 'engine/run_engine.py'], check=True)
subprocess.run([sys.executable, 'models/train.py'], check=True)

## 4. Generate the three scenario cards for Bengaluru
(ΔNDVI from the model's own forecast; prompts are auditable in the output JSON)

In [ ]:
env = dict(os.environ, EP_SCENARIO_IMAGES='1')
proc = subprocess.run([sys.executable, 'models/scenario_cards.py', '--generate',
                      '--lat', '12.97', '--lon', '77.59', '--parameter', 'ndvi'],
                     capture_output=True, text=True, env=env, check=False)
print(proc.stdout[-4000:])
print(proc.stderr[-2000:])

## 5. Look at them

In [ ]:
from IPython.display import Image, display
pngs = sorted(glob.glob('/content/earth-pulse/models/artifacts/scenarios/*.png'))
print('generated:', pngs)
for p in pngs:
    display(Image(p))

In [ ]:
with zipfile.ZipFile('/content/earth-pulse-scenarios.zip', 'w') as zf:
    for p in pngs: zf.write(p, os.path.basename(p))
print('zip ready — use the Files pane (left) to download earth-pulse-scenarios.zip')

## Troubleshooting
- **No GPU attached** → Runtime → Change runtime type → GPU (T4). Re-run all cells.
- **OOM / model too big** → set `EP_SCENARIO_MODEL` to a smaller SD variant, e.g. `runwayml/stable-diffusion-v1-5` is already the default (1.7 GB); check `nvidia-smi`.
- **Slow download** → wait; the first run downloads ~4 GB. Re-runs reuse Colab's cache.
- **429 / rate limit** → retry later or switch model ID via `EP_SCENARIO_MODEL=...`.
- **Images look off** → strength is per-variant (lower 0.25 / median 0.35 / upper 0.45). That's the conservative band by design.

### Pushing to Hugging Face Hub (optional, for the live demo)
```python
!huggingface-cli login        # paste your token (gets typed in the form, not in logs)
!huggingface-cli upload <you>/earth-pulse-scenarios /content/earth-pulse/models/artifacts/scenarios
```